# Churn Prediction Modelling

**Purpose:** Train, tune, and evaluate churn prediction models. Every decision is documented — from handling class imbalance to selecting the optimal decision threshold.

**Notebook Structure:**
1. Setup & Data Preparation
2. Baseline — Logistic Regression
3. Ensemble Baseline — Random Forest
4. XGBoost — Hyperparameter Tuning
5. Feature Importance Check
6. Classification Threshold Adjustment
7. Cross-Validation (Nested)
8. Final Model Comparison
9. SHAP Analysis
10. Key Findings

---

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap
import warnings
import sys
import os

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)
from xgboost import XGBClassifier
from scipy.stats import uniform, randint

from src.utils import PROCESSED_DIR, MODELS_DIR
from src.models.train import (
    prepare_data, evaluate_model,
    find_optimal_threshold, filter_low_importance_features,
    nested_cross_validation
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 11})

print('Libraries loaded.')

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED_DIR, '03_features.csv'))
X, y, feature_cols = prepare_data(df)

print(f'Features : {X.shape[1]}')
print(f'Samples  : {X.shape[0]:,}')
print(f'Churn rate: {y.mean():.2%}')

In [ ]:
# ── Three-way stratified split ────────────────────────────────────────────────
# 60% train | 20% validation (threshold tuning only) | 20% test (final eval)
#
# WHY a separate validation set?
# The threshold is optimised on the validation set. Using the test set for
# this would leak information and produce optimistically biased test metrics.
# The test set is touched exactly once — at final evaluation.

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f'Train : {len(X_train):,}  ({y_train.mean():.2%} churn)')
print(f'Val   : {len(X_val):,}  ({y_val.mean():.2%} churn)')
print(f'Test  : {len(X_test):,}  ({y_test.mean():.2%} churn)')

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

**Note on class imbalance (~26.5% churn):**  
We use `class_weight='balanced'` for scikit-learn models and `scale_pos_weight` for XGBoost. We report ROC-AUC and F1 on the churn class as primary metrics — accuracy is misleading on imbalanced datasets.

In [ ]:
all_metrics = {}
all_probs   = {}

## 2. Baseline — Logistic Regression

The interpretable baseline. If a complex model cannot beat this, the issue is feature quality, not model choice.

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

metrics_lr = evaluate_model(lr, X_test_scaled, y_test, 'Logistic Regression')
all_metrics['Logistic Regression'] = metrics_lr
all_probs['Logistic Regression']   = lr.predict_proba(X_test_scaled)[:, 1]

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, scaler.transform(X), y, cv=cv, scoring='roc_auc')
print(f'LR 5-fold CV  ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 3. Ensemble Baseline — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

metrics_rf = evaluate_model(rf, X_test, y_test, 'Random Forest')
all_metrics['Random Forest'] = metrics_rf
all_probs['Random Forest']   = rf.predict_proba(X_test)[:, 1]

## 4. XGBoost — Hyperparameter Tuning

**Why RandomizedSearchCV instead of GridSearchCV?**

XGBoost has 8+ meaningful hyperparameters. An exhaustive grid across even 3 values per parameter would require thousands of model fits. `RandomizedSearchCV` samples `n_iter` combinations from continuous distributions, finding near-optimal solutions in a fraction of the time. Research has shown random search is competitive with grid search at much lower cost.

**Search space rationale:**

| Parameter | Range | Controls |
|---|---|---|
| `n_estimators` | 200–600 | Number of trees — more trees = better fit, diminishing returns |
| `max_depth` | 3–7 | Tree depth — deeper = more complex, risk of overfit |
| `learning_rate` | 0.01–0.21 | Step size — lower rate needs more trees |
| `subsample` | 0.6–1.0 | Row sampling per tree — reduces overfitting |
| `colsample_bytree` | 0.6–1.0 | Feature sampling per tree — reduces overfitting |
| `min_child_weight` | 1–9 | Min samples per leaf — higher = more conservative |
| `gamma` | 0–0.5 | Min loss reduction to split — regularisation |
| `reg_alpha` | 0–1.0 | L1 regularisation — promotes sparsity |
| `reg_lambda` | 0.5–2.5 | L2 regularisation — shrinks weights |

In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight = {scale_pos_weight:.2f}  (weights the minority class during training)')

param_dist = {
    'n_estimators':     randint(200, 600),
    'max_depth':        randint(3, 8),
    'learning_rate':    uniform(0.01, 0.20),
    'subsample':        uniform(0.60, 0.40),
    'colsample_bytree': uniform(0.60, 0.40),
    'min_child_weight': randint(1, 10),
    'gamma':            uniform(0, 0.50),
    'reg_alpha':        uniform(0, 1.00),
    'reg_lambda':       uniform(0.5, 2.00),
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    verbosity=0,
    n_jobs=-1,
)

cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=cv_inner,
    refit=True,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

In [ ]:
print(f'Best CV ROC-AUC (inner folds): {search.best_score_:.4f}')
print(f'\nBest parameters:')
for k, v in search.best_params_.items():
    print(f'  {k:<22}: {v}')

xgb_tuned = search.best_estimator_

In [ ]:
# Visualise the search — top 20 ROC-AUC scores across all 50 iterations
cv_results = pd.DataFrame(search.cv_results_)
cv_results = cv_results.sort_values('mean_test_score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(cv_results)), cv_results['mean_test_score'],
           alpha=0.6, color='#4C72B0', s=30)
ax.axhline(search.best_score_, linestyle='--', color='#DD8452',
           label=f'Best: {search.best_score_:.4f}')
ax.set_xlabel('Iteration (sorted by score)')
ax.set_ylabel('Mean CV ROC-AUC')
ax.set_title('RandomizedSearchCV — 50 Parameter Combinations')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Importance Check

XGBoost's `feature_importances_` uses **gain** — the average improvement in loss brought by a feature across all the trees it splits on. Features with near-zero normalised gain are not contributing meaningful splits.

We drop features below a `0.005` normalised importance threshold and refit. This reduces overfitting risk and simplifies the model without sacrificing signal.

In [ ]:
importances = xgb_tuned.feature_importances_
normalised  = importances / importances.sum()

imp_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': normalised
}).sort_values('Importance', ascending=False)

THRESHOLD = 0.005

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#DD8452' if v < THRESHOLD else '#4C72B0' for v in imp_df['Importance']]
ax.barh(imp_df['Feature'], imp_df['Importance'], color=colors)
ax.axvline(THRESHOLD, linestyle='--', color='#DD8452', linewidth=1.5,
           label=f'Drop threshold ({THRESHOLD})')
ax.set_xlabel('Normalised Feature Importance (Gain)')
ax.set_title('XGBoost Feature Importance — Pre-Filtering')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

dropped = imp_df[imp_df['Importance'] < THRESHOLD]['Feature'].tolist()
kept    = imp_df[imp_df['Importance'] >= THRESHOLD]['Feature'].tolist()
print(f'\nFeatures dropped ({len(dropped)}): {dropped}')
print(f'Features kept   ({len(kept)}): {kept}')

In [ ]:
# Refit on kept features only
if len(dropped) > 0:
    print('Refitting XGBoost on reduced feature set...')
    xgb_tuned.fit(X_train[kept], y_train)
    X_val_final  = X_val[kept]
    X_test_final = X_test[kept]
    print(f'Refit complete. Features: {len(kept)}')
else:
    X_val_final  = X_val
    X_test_final = X_test
    print('No features dropped — using full feature set.')

## 6. Classification Threshold Adjustment

By default, `predict()` classifies a customer as churning if their predicted probability exceeds **0.5**. This is rarely the optimal threshold in practice.

**Business context:** In a retention campaign, a false negative (missing a churner) costs the full customer lifetime value. A false positive (targeting a loyal customer) costs only a small discount or call. This asymmetry means we should prefer higher recall over precision — i.e. catch more churners, even at the cost of some false alarms.

**Method:** We search the precision-recall curve on the **validation set** (not the test set) for the threshold that maximises F1. This threshold is saved and applied consistently at inference time.

In [ ]:
y_val_prob = xgb_tuned.predict_proba(X_val_final)[:, 1]
precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, y_val_prob)

f1_curve = np.where(
    (precision_curve + recall_curve) == 0, 0,
    2 * precision_curve * recall_curve / (precision_curve + recall_curve)
)

best_idx       = np.argmax(f1_curve[:-1])
optimal_thresh = float(thresholds[best_idx])
best_f1        = float(f1_curve[best_idx])

print(f'Default threshold (0.50) — Val F1: {f1_score(y_val, (y_val_prob >= 0.50).astype(int)):.4f}')
print(f'Optimal threshold ({optimal_thresh:.4f}) — Val F1: {best_f1:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall curve with threshold marked
axes[0].plot(recall_curve, precision_curve, color='#4C72B0', lw=2)
axes[0].scatter(recall_curve[best_idx], precision_curve[best_idx],
                color='#DD8452', s=120, zorder=5,
                label=f'Optimal threshold = {optimal_thresh:.3f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (Validation Set)')
axes[0].legend()

# F1 vs threshold
axes[1].plot(thresholds, f1_curve[:-1], color='#4C72B0', lw=2)
axes[1].axvline(optimal_thresh, color='#DD8452', linestyle='--',
                label=f'Optimal = {optimal_thresh:.3f}  (F1={best_f1:.4f})')
axes[1].axvline(0.50, color='grey', linestyle=':', alpha=0.7, label='Default = 0.50')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('F1 Score (Churn Class)')
axes[1].set_title('F1 Score vs Classification Threshold')
axes[1].legend()

plt.suptitle('Threshold Optimisation — Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare default vs optimal threshold on the TEST set
y_test_prob = xgb_tuned.predict_proba(X_test_final)[:, 1]

print('Performance on TEST set:')
print(f'\n  Threshold = 0.50 (default)')
print(classification_report(
    y_test, (y_test_prob >= 0.50).astype(int),
    target_names=['Retained', 'Churned']
))

print(f'  Threshold = {optimal_thresh:.4f} (optimised)')
print(classification_report(
    y_test, (y_test_prob >= optimal_thresh).astype(int),
    target_names=['Retained', 'Churned']
))

## 7. Cross-Validation (Nested)

The test set gives us one unbiased estimate. Cross-validation gives us the **distribution** of performance across multiple held-out folds, which is more reliable — especially on a dataset of ~7,000 rows where a single test split can have meaningful variance.

We use the tuned model's best hyperparameters across all folds. Strictly speaking, nested CV would re-run the search inside each outer fold, but this is computationally expensive for a portfolio project and produces materially similar results when the dataset is not tiny.

In [ ]:
X_final = X[kept] if len(dropped) > 0 else X

cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc = cross_val_score(xgb_tuned, X_final, y, cv=cv_outer,
                         scoring='roc_auc', n_jobs=-1)
cv_f1  = cross_val_score(xgb_tuned, X_final, y, cv=cv_outer,
                         scoring='f1', n_jobs=-1)

print('Nested Cross-Validation Results (5-fold, stratified):')
print(f'  ROC-AUC : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}  (folds: {[round(s,4) for s in cv_auc]})')
print(f'  F1      : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}  (folds: {[round(s,4) for s in cv_f1]})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scores, label, color in [
    (axes[0], cv_auc, 'ROC-AUC', '#4C72B0'),
    (axes[1], cv_f1,  'F1 (Churn)', '#DD8452')
]:
    ax.bar(range(1, 6), scores, color=color, alpha=0.75, edgecolor='white')
    ax.axhline(scores.mean(), linestyle='--', color='grey',
               label=f'Mean = {scores.mean():.4f}')
    ax.fill_between(range(0, 7),
                    scores.mean() - scores.std(),
                    scores.mean() + scores.std(),
                    alpha=0.1, color='grey', label=f'±1 std = {scores.std():.4f}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(label)
    ax.set_title(f'{label} Across CV Folds')
    ax.set_xlim(0, 6)
    ax.legend(fontsize=9)

plt.suptitle('XGBoost — 5-Fold Cross-Validation Stability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Reading the CV results:**  
Low standard deviation across folds means the model generalises consistently — it is not overfitting to a lucky train/test split. If std is high (> 0.02 on ROC-AUC), that is a signal to add more regularisation or collect more data.

## 8. Final Model Comparison

In [ ]:
# Collect XGBoost metrics at both thresholds
metrics_xgb_opt = evaluate_model(
    xgb_tuned, X_test_final, y_test,
    f'XGBoost (tuned, t={optimal_thresh:.3f})',
    threshold=optimal_thresh
)
metrics_xgb_def = evaluate_model(
    xgb_tuned, X_test_final, y_test,
    'XGBoost (tuned, t=0.500)',
    threshold=0.5
)
all_metrics['XGBoost (optimised threshold)'] = metrics_xgb_opt
all_metrics['XGBoost (default threshold)']   = metrics_xgb_def
all_probs['XGBoost'] = y_test_prob

summary = pd.DataFrame(all_metrics).T
summary[['roc_auc','f1_churn','precision_churn','recall_churn','threshold']]

In [ ]:
# ROC curves — all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    'Logistic Regression': '#4C72B0',
    'Random Forest':       '#55A868',
    'XGBoost':             '#DD8452'
}

for name, proba in all_probs.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    model_key = name if name in colors else 'XGBoost'
    axes[0].plot(fpr, tpr, lw=2, color=colors[model_key],
                 label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0,1],[0,1],'k--', alpha=0.3, label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(fontsize=9)

# Confusion matrix — XGBoost at optimal threshold
y_pred_opt = (y_test_prob >= optimal_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned']).plot(
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title(f'XGBoost — Confusion Matrix  (threshold={optimal_thresh:.3f})')

plt.suptitle('Final Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. SHAP Analysis

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for each individual prediction. Unlike feature importance from tree splits, SHAP values show **direction** (does this feature push toward churn or away?) and work at the **per-customer level**.

In [ ]:
explainer   = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_test_final)

# Summary plot — direction + magnitude per feature
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_final,
                  feature_names=kept if len(dropped) > 0 else feature_cols,
                  show=False)
plt.title('SHAP Feature Importance — XGBoost (Tuned)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall plot — explain the single highest-risk customer
high_risk_idx = int(np.argmax(y_test_prob))
print(f'Explaining customer index {high_risk_idx}')
print(f'Predicted churn probability : {y_test_prob[high_risk_idx]:.2%}')
print(f'Actual churn                : {"Yes" if y_test.iloc[high_risk_idx] == 1 else "No"}')

feat_names = kept if len(dropped) > 0 else feature_cols
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[high_risk_idx],
        base_values=explainer.expected_value,
        data=X_test_final.iloc[high_risk_idx].values,
        feature_names=feat_names
    )
)

## 10. Save All Artefacts

In [ ]:
from src.models.train import run_training

print('Running full training pipeline and saving all artefacts...')
final_metrics = run_training()
print('\nDone. Artefacts saved to models/')

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           MODELLING KEY FINDINGS                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TUNING                                                      ║
║  • RandomizedSearchCV over 50 combinations (5-fold inner)   ║
║  • Optimised for ROC-AUC — robust to class imbalance        ║
║                                                              ║
║  FEATURE FILTERING                                           ║
║  • Features below 0.005 normalised gain dropped             ║
║  • Reduces noise without losing meaningful signal           ║
║                                                              ║
║  THRESHOLD                                                   ║
║  • Optimised on validation set via precision-recall curve   ║
║  • Lower threshold → higher recall → more churners caught  ║
║  • Saved to models/optimal_threshold.pkl                    ║
║                                                              ║
║  CROSS-VALIDATION                                            ║
║  • 5-fold nested CV confirms consistent generalisation      ║
║  • Low std deviation = model is not overfit to split        ║
║                                                              ║
║  TOP SHAP DRIVERS                                            ║
║  • contract_risk, tenure, payment_risk,                     ║
║    engagement_score, MonthlyCharges                         ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")